# Rewriting bytecode while the program runs

*How does the interpreter rewrite bytecode while the program is running?*

This needs Python 3.11 or newer. The outputs below are from Python 3.14.3.

In [1]:
import dis, sys

print(sys.version)

3.14.3 (main, Feb  3 2026, 15:32:20) [Clang 15.0.0 (clang-1500.1.0.2.5)]


Before `add` has ever run, its `+` is the general `BINARY_OP`. Each time it runs, it has to check what `a` and `b` are and find the right `__add__`.

In [2]:
def add(a, b):
    return a + b

dis.dis(add, adaptive=True)

  1           RESUME                   0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE


Call it once with ints, and look again.

In [3]:
add(1, 2)
dis.dis(add, adaptive=True)

  1           RESUME_CHECK             0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE


Once more.

In [4]:
add(1, 2)
dis.dis(add, adaptive=True)

  1           RESUME_CHECK             0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


Python rewrote the instruction in place. `BINARY_OP_ADD_INT` bets that both operands are ints and skips the lookup. (`RESUME`, at the top of the function, already became `RESUME_CHECK` on the first call.)

Now strings:

In [5]:
print(add("a", "b"))
dis.dis(add, adaptive=True)

ab
  1           RESUME_CHECK             0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


Still `ADD_INT`. The bet missed, so this call fell back to the general `+`, and it still returned `"ab"`. One miss isn't enough to make Python give up on a bet. How many does it take?

In [6]:
calls = 1
while "ADD_UNICODE" not in dis.Bytecode(add, adaptive=True).dis():
    add("a", "b")
    calls += 1

print(calls, "string calls")
dis.dis(add, adaptive=True)

53 string calls
  1           RESUME_CHECK             0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP_ADD_UNICODE    0 (+)
              RETURN_VALUE


Without `adaptive=True`, `dis` shows the instructions the compiler wrote. Those never change: the rewrites happen in the copy the interpreter actually runs.

In [7]:
dis.dis(add)

  1           RESUME                   0

  2           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (a, b)
              BINARY_OP                0 (+)
              RETURN_VALUE
